In [1]:
import torch
import torchtext
import numpy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils import data
import pandas as pd
# jieba分词
import jieba


In [2]:
data = pd.read_csv('waimai_10k.csv')

In [3]:
data.head()

,label,review
0,1,很快，好吃，味道足，量大
1,1,没有送水没有送水没有送水
2,1,非常快，态度好。
3,1,方便，快捷，味道可口，快递给力
4,1,菜味道很棒！送餐很及时！


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11987 entries, 0 to 11986
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   11987 non-null  int64 
 1   review  11987 non-null  object
dtypes: int64(1), object(1)
memory usage: 187.4+ KB


In [5]:
data.label.value_counts()

label
0    7987
1    4000
Name: count, dtype: int64

In [6]:
def pre_text(text):
    text = text.replace('，','').replace('！','').replace('。','').replace('“','').replace('”','')
    return jieba.lcut(text)

In [7]:
# text='0,虽说是周六，可能是点餐早的缘故，送餐速度挺快的。可是餐品的内容就不敢恭维了，小到“没朋友”的油条和三个南瓜饼。不是一般的坑人！比肯德基性价比还差。'

In [8]:
# pre_text(text)

In [9]:
data['review'] = data.review.apply(pre_text)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\rehuanoo\AppData\Local\Temp\jieba.cache
Loading model cost 0.309 seconds.
Prefix dict has been built successfully.


In [10]:
data.head()

,label,review
0,1,"[很快, 好吃, 味道, 足量, 大]"
1,1,"[没有, 送水, 没有, 送水, 没有, 送水]"
2,1,"[非常, 快, 态度, 好]"
3,1,"[方便快捷, 味道, 可口, 快, 递给, 力]"
4,1,"[菜, 味道, 很棒, 送餐, 很, 及时]"


In [11]:
# 用于创建词汇表
from torchtext.vocab import build_vocab_from_iterator

def yield_tokens(data):
    for text in data:
        yield text

vocab = build_vocab_from_iterator(yield_tokens(data.review),specials=['<pad>','<unk>'],min_freq=2)
# 创建词汇表默认索引,min_freq=2 最小词频,至少出现两次
vocab.set_default_index(vocab['<unk>'])

In [12]:
vocab_size = len(vocab)
vocab_size

5441

In [13]:
vocab(data.review[0])

[55, 14, 13, 5226, 114]

In [14]:
vocab(['我','爱','你'])

[8, 666, 82]

In [15]:
# 80%训练，20%测试
train_num = int(len(data)*0.8)
train_num

9589

In [16]:
test_num = len(data)-train_num
test_num

2398

In [17]:
train_data = data.sample(train_num)
train_data.head()

,label,review
2637,1,"[快递, 小哥, 太棒了, 哈哈, ,, 菜, 也, 好吃]"
1653,1,"[除了, 速度, 有待, 加强, 其他, 都, 不错]"
1886,1,"[还, 可以, 就是, 等, 得, 太久, 太久, 了, 两个, 小时]"
9961,0,"[拌, 饭, 咋, 没有, 鸡蛋, 呢, 严重, 怀疑, 给, 我, 少, 送, 了, 不爽]"
2051,1,"[很, 好吃, 肉, 很多, 很, 实惠]"


In [18]:
data.index

RangeIndex(start=0, stop=11987, step=1)

In [19]:
train_data.index

Index([ 2637,  1653,  1886,  9961,  2051,  6640,  6203, 11049,  5034,  4294,
       ...
        8242,  7288,  2406,  1725, 11532, 10005,  2096, 11862,  1704, 10898],
      dtype='int64', length=9589)

In [20]:
test_data = data.iloc[data.index[~data.index.isin(train_data.index)]]
test_data

,label,review
0,1,"[很快, 好吃, 味道, 足量, 大]"
1,1,"[没有, 送水, 没有, 送水, 没有, 送水]"
12,1,"[味道, 好, 送, 餐快, 分量, 足]"
21,1,"[送货, 速度, 很快, 一直, 定, 这家, 赞]"
23,1,"[好吃, 总点, 这么, 多够, 五个, 人, 吃, 送, 的, 很快]"
...,...,...
11938,0,"[外卖, 骑士, 将近, 两, 小时, 送到, ,, 而且, 还要, 我, 下去, 取餐]"
11943,0,"[绿豆汤, 馊, 了, 我, 明天, 铁定, 拉肚子]"
11946,0,"[味道, 依然, 很, 好, 送餐, 等, 了, 一个半, 小时, …]"
11947,0,"[吃, 出, 一个, 毛发, ,, 和, 一堆, 不, 知道, 是, 什么, 的, 东西]"


In [21]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)
def collate_batch(batch):
    label_list,text_list=[],[]
    for (_label,_text) in batch:
        label_list.append(_label)
        precess_text = torch.tensor(vocab(_text),dtype=torch.int64)
        text_list.append(precess_text)
    label_list = torch.tensor(label_list)
    text_list = torch.nn.utils.rnn.pad_sequence(text_list)

    return label_list.to(device),text_list.to(device)

cuda


In [22]:
from torch.utils.data import DataLoader
BATCH_SIZE = 64
# 数据加载器
train_dl = torch.utils.data.DataLoader(
    train_data.values,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_batch
)
test_dl = torch.utils.data.DataLoader(
    test_data.values,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_batch
)

In [23]:
# idx,(labels,text) = next(enumerate(train_dl))

In [24]:
# text.size()

In [25]:
# 创建LSTM循环神经网络单元
hidden_size = 128
class RNN_Encoder(nn.Module):
    # 输入的维度，隐藏层的大小
    def __init__(self,input_dim,hidden_size):
        super(RNN_Encoder,self).__init__()
        self.rnn_cell = nn.LSTMCell(input_dim,hidden_size)
    def forward(self,inputs):
        # input输入数据的形状sequence_length,batch_size,input_dim
        bz = inputs.shape[1]
        # ht为隐藏装动态
        ht = torch.zeros((bz,hidden_size)).to(device)
        # 增加单元隐藏状态
        ct = torch.zeros((bz,hidden_size)).to(device)
        for word in inputs:
            ht,ct = self.rnn_cell(word,(ht,ct))
        return ht



In [26]:
# class RNN_Net(nn.Module):
#     def __init__(self,vocab_size,embeding_dim,hidden_size):
#         super(RNN_Net,self).__init__()
#         # 创建一个嵌入层，固定输入的维度
#         # 一段文字-》词汇表对应的索引-》转换为嵌入向量
#         self.em = nn.Embedding(vocab_size,embeding_dim)
#         # 实例化循环神经网络
#         self.rnn = RNN_Encoder(embeding_dim,hidden_size)
#         # 实例化全连接层
#         self.fc1 = nn.Linear(hidden_size,64)
#         self.fc2 = nn.Linear(64,2)

#     def forward(self,inputs):
#         x = self.em(inputs)
#         x = self.rnn(x)
#         x = self.fc1(x)
#         x = F.relu(x)
#         x = self.fc2(x)
#         return x


In [29]:
class RNN_Net(nn.Module):
    def __init__(self,vocab_size,embeding_dim,hidden_size):
        super(RNN_Net,self).__init__()
        # 创建一个嵌入层，固定输入的维度
        # 一段文字-》词汇表对应的索引-》转换为嵌入向量
        self.em = nn.Embedding(vocab_size,embeding_dim)
        # 实例化双向循环神经网络
        self.rnn = nn.LSTM(embeding_dim,hidden_size,bidirectional=True)
        # 实例化全连接层
        self.fc1 = nn.Linear(hidden_size*2,64)
        self.fc2 = nn.Linear(64,2)

    def forward(self,inputs):
        x = self.em(inputs)
        x = F.dropout(x)
        # 通过双向lstm，返回隐藏状态和细胞状态
        x,_ = self.rnn(x)
        x = self.fc1(x[-1])
        x = F.relu(x)
        x = F.dropout(x)
        x = self.fc2(x)
        return x


In [30]:
vocab_size = len(vocab)
emsize = 300
num_class=2
model = RNN_Net(vocab_size,emsize,128)
model = model.to(device)

In [31]:
model

RNN_Net(
  (em): Embedding(5441, 300)
  (rnn): LSTM(300, 128, bidirectional=True)
  (fc1): Linear(in_features=256, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)

In [32]:
from torch.optim import lr_scheduler
loss_fn = nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

exp_lr_scheduler = lr_scheduler.StepLR(optimizer,step_size=5,gamma=0.1)

In [33]:
def train(dataloader):
    total_acc,total_count,total_loss=0,0,0
    # 将模型设置训练模式
    model.train()
    for label,text in dataloader:
        predited_label = model(text)
        loss = loss_fn(predited_label,label)

        optimizer.zero_grad()
        loss.backward()
        # for name, param in model.named_parameters():
        #     if param.grad is not None:
        #         print(name, param.grad.norm())
        optimizer.step()
        with torch.no_grad():
            total_acc += (predited_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
            total_loss += loss.item()*label.size(0)
    return total_loss/total_count,total_acc/total_count
            

In [34]:
def test(dataloader):
    total_acc,total_count,total_loss=0,0,0
    # 将模型设置评估模式
    model.eval()
    with torch.no_grad():
        for idx,(label,text) in enumerate(dataloader):
            predited_label = model(text)
            loss = loss_fn(predited_label,label)
            
            total_acc += (predited_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
            total_loss += loss.item()*label.size(0)
    return total_loss/total_count,total_acc/total_count

In [35]:
# 训练多轮
epochs = 10
train_acc=[]
test_acc=[]
train_loss=[]
test_loss=[]

# 保存最佳的准确率
best_acc = 0.0
best_acc_index=0
for epoch in range(epochs):
    epoch_loss,epoch_acc = train(train_dl)
    epoch_test_loss,epoch_test_acc = test(test_dl)
    train_acc.append(epoch_acc)
    test_acc.append(epoch_test_acc)
    train_loss.append(epoch_loss)
    test_loss.append(epoch_test_loss)
    exp_lr_scheduler.step()
    print('epoch:{},train_loss:{},train_acc{}'.format(epoch,epoch_loss,epoch_acc))
    print('epoch:{},test_loss:{},test_acc{}'.format(epoch,epoch_test_loss,epoch_test_acc))
    print('---------------------------------')
    # if epoch_test_acc > best_acc:
    #     print('保存最佳准确率模型',epoch_test_acc)
    #     best_acc = epoch_test_acc
    #     best_acc_index = best_acc_index+1
    #     # 保存模型结构，包含参数
    #     torch.save(model,'best_model.pth')
    #     # 保存参数
    #     # torch.save(model1.state_dict(),'best_model_weights'+str(best_acc_index)+'.pth')

epoch:0,train_loss:0.6436077745332001,train_acc0.6663885702367296
epoch:0,test_loss:0.6391859142058486,test_acc0.6655546288573811
---------------------------------
epoch:1,train_loss:0.6372716132338148,train_acc0.666492856397956
epoch:1,test_loss:0.6284159300623584,test_acc0.6655546288573811
---------------------------------
epoch:2,train_loss:0.5050141240934073,train_acc0.7565960996975701
epoch:2,test_loss:0.41517852852004483,test_acc0.8256880733944955
---------------------------------
epoch:3,train_loss:0.3720403206287503,train_acc0.8524350818646366
epoch:3,test_loss:0.36193411746951715,test_acc0.853628023352794
---------------------------------
epoch:4,train_loss:0.3247588603510724,train_acc0.8785066221712379
epoch:4,test_loss:0.35006956598279476,test_acc0.8636363636363636
---------------------------------
epoch:5,train_loss:0.2878086708213356,train_acc0.8956095526123683
epoch:5,test_loss:0.35256246639550776,test_acc0.8665554628857381
---------------------------------
epoch:6,train_